In [ ]:
# ============================================================================
# 02_bronze_to_silver  (Airflow Option-B port)
# ----------------------------------------------------------------------------
# Parameter contract injected by the DAG's `bronze_to_silver` mapped task
# (build_bronze_to_silver_params): run_id_root + the cohort fields + stage.
# run_id is derived identically to the generate stage so the silver MERGE reads
# exactly the path that generate wrote.
# ============================================================================

# PARAMETERS CELL ************
run_id_root = ""              # run grouping key from set_run_id
dataset_id  = "ma_diabetes"   # logical cohort name (= cohort_id)
state       = "Massachusetts" # passed through; unused by this stage
stage       = "bronze_to_silver"


In [ ]:
# ---------------------------------------------------------------------------
# run_state skip/restart helpers (gold.control.run_state)
# Paste-in contract from include/run_state_helpers.py. The DAG never queries the
# lakehouse; each notebook self-records RUNNING -> SUCCEEDED/FAILED and self-skips
# a (run_id_root, cohort_id, stage) triple that already SUCCEEDED.
# ---------------------------------------------------------------------------
from datetime import datetime, timezone

RUN_STATE_TABLE = "lh_synthea_gold.control.run_state"


def already_succeeded(run_id_root, cohort_id, stage):
    """True if this (run_id_root, cohort_id, stage) already completed."""
    if not run_id_root or not spark.catalog.tableExists(RUN_STATE_TABLE):
        return False
    df = spark.sql(
        f"""
        SELECT 1 FROM {RUN_STATE_TABLE}
        WHERE run_id_root = '{run_id_root}'
          AND cohort_id   = '{cohort_id}'
          AND stage       = '{stage}'
          AND status      = 'SUCCEEDED'
        LIMIT 1
        """
    )
    return df.count() > 0


def mark(run_id_root, cohort_id, stage, status, error=None):
    """Idempotent UPSERT of a run_state row for this triple via MERGE."""
    if not run_id_root or not spark.catalog.tableExists(RUN_STATE_TABLE):
        return
    now = datetime.now(timezone.utc)
    err = (error or "").replace("'", "''")[:4000]
    spark.sql(
        f"""
        MERGE INTO {RUN_STATE_TABLE} AS t
        USING (
            SELECT
                '{run_id_root}' AS run_id_root,
                '{cohort_id}'   AS cohort_id,
                '{stage}'       AS stage,
                '{status}'      AS status,
                TIMESTAMP('{now.isoformat()}') AS ts,
                '{err}'         AS error
        ) AS s
        ON  t.run_id_root = s.run_id_root
        AND t.cohort_id   = s.cohort_id
        AND t.stage       = s.stage
        WHEN MATCHED THEN UPDATE SET
            t.status   = s.status,
            t.attempt  = COALESCE(t.attempt, 0) + CASE WHEN s.status = 'RUNNING' THEN 1 ELSE 0 END,
            t.started_ts = CASE WHEN s.status = 'RUNNING' THEN s.ts ELSE t.started_ts END,
            t.ended_ts   = CASE WHEN s.status IN ('SUCCEEDED','FAILED') THEN s.ts ELSE t.ended_ts END,
            t.error      = CASE WHEN s.status = 'FAILED' THEN s.error ELSE NULL END
        WHEN NOT MATCHED THEN INSERT (
            run_id_root, cohort_id, stage, status, attempt, started_ts, ended_ts, error
        ) VALUES (
            s.run_id_root, s.cohort_id, s.stage, s.status, 1, s.ts, NULL,
            CASE WHEN s.status = 'FAILED' THEN s.error ELSE NULL END
        )
        """
    )

import json

cohort_id = dataset_id
run_id    = f"{run_id_root}-{dataset_id}" if run_id_root else f"interactive-{dataset_id}"

if already_succeeded(run_id_root, cohort_id, stage):
    mssparkutils.notebook.exit(json.dumps({"status": "SKIPPED", "run_id": run_id}))
mark(run_id_root, cohort_id, stage, "RUNNING")


In [ ]:
# The original stage body is wrapped so that a thrown exception is recorded
# as FAILED in run_state, while a normal/early return is recorded SUCCEEDED.
def _run_body():
    # CODE CELL ******************
    import os
    from pyspark.sql import SparkSession, functions as F
    from delta.tables import DeltaTable

    spark = SparkSession.builder.getOrCreate()

    if not run_id:
        raise ValueError("run_id parameter is required")

    cohort_id = dataset_id
    SILVER_DB = "lh_synthea_silver"
    SCHEMA    = "core"

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_DB}.{SCHEMA}")

    # CSV directory in bronze (mounted as ABFSS via OneLake)
    bronze_csv_dir = (
        f"abfss://synthea-data-airflow-ws@onelake.dfs.fabric.microsoft.com/"
        f"lh_synthea_bronze.Lakehouse/Files/raw/{dataset_id}/{run_id}/csv"
    )

    # Table name -> (csv filename, list of natural-key columns excluding cohort_id)
    TABLES = {
        "patients":            ("patients.csv",            ["Id"]),
        "encounters":          ("encounters.csv",          ["Id"]),
        "organizations":       ("organizations.csv",       ["Id"]),
        "providers":           ("providers.csv",           ["Id"]),
        "payers":              ("payers.csv",              ["Id"]),
        "claims":              ("claims.csv",              ["Id"]),
        "claims_transactions": ("claims_transactions.csv", ["Id"]),
        "imaging_studies":     ("imaging_studies.csv",     ["Id"]),
        "careplans":           ("careplans.csv",           ["Id"]),
        "conditions":          ("conditions.csv",          ["PATIENT", "ENCOUNTER", "CODE", "START"]),
        "observations":        ("observations.csv",        ["PATIENT", "ENCOUNTER", "CODE", "DATE"]),
        "procedures":          ("procedures.csv",          ["PATIENT", "ENCOUNTER", "CODE", "START"]),
        "medications":         ("medications.csv",         ["PATIENT", "ENCOUNTER", "CODE", "START"]),
        "immunizations":       ("immunizations.csv",       ["PATIENT", "ENCOUNTER", "CODE", "DATE"]),
        "allergies":           ("allergies.csv",           ["PATIENT", "ENCOUNTER", "CODE", "START"]),
        "devices":             ("devices.csv",             ["PATIENT", "ENCOUNTER", "CODE", "START"]),
        "supplies":            ("supplies.csv",            ["PATIENT", "ENCOUNTER", "CODE", "DATE"]),
        "payer_transitions":   ("payer_transitions.csv",   ["PATIENT", "PAYER", "START_DATE"]),
    }

    def _table_exists(fqn: str) -> bool:
        return spark.catalog.tableExists(fqn)

    def _read_csv(path: str):
        return (
            spark.read
                 .option("header",        "true")
                 .option("inferSchema",   "false")
                 .option("multiLine",     "true")
                 .option("escape",        '"')
                 .option("nullValue",     "")
                 .csv(path)
        )

    # Medical codes / identifiers vary in width across cohorts (some SNOMED codes
    # exceed INT range, LOINC codes contain hyphens). Reading every column as
    # string keeps the silver schema identical for all cohorts so MERGE never hits
    # a type mismatch or overflow. Only the money columns that the gold layer sums
    # are coerced to double.
    NUMERIC_COLS = {"TOTAL_CLAIM_COST", "PAYMENTS", "OUTSTANDING"}

    def _coerce_numeric(df):
        for c in df.columns:
            if c.upper() in NUMERIC_COLS:
                df = df.withColumn(c, F.col(c).cast("double"))
        return df

    def _stamp(df, cohort, run, src_file):
        return (df
                .withColumn("cohort_id",   F.lit(cohort))
                .withColumn("run_id",      F.lit(run))
                .withColumn("load_ts",     F.current_timestamp())
                .withColumn("source_file", F.lit(src_file)))

    def _merge(df, fqn, key_cols):
        if not _table_exists(fqn):
            (df.write
               .format("delta")
               .mode("overwrite")
               .option("delta.autoOptimize.optimizeWrite", "true")
               .saveAsTable(fqn))
            return "created"

        target = DeltaTable.forName(spark, fqn)
        full_keys = ["cohort_id"] + key_cols
        cond = " AND ".join([f"t.`{c}` = s.`{c}`" for c in full_keys])
        (target.alias("t")
               .merge(df.alias("s"), cond)
               .whenMatchedUpdateAll()
               .whenNotMatchedInsertAll()
               .execute())
        return "merged"

    results = []
    for tname, (fname, keys) in TABLES.items():
        fqn  = f"{SILVER_DB}.{SCHEMA}.{tname}"
        path = f"{bronze_csv_dir}/{fname}"
        try:
            # Check existence via mssparkutils (works for OneLake ABFSS)
            exists = True
            try:
                mssparkutils.fs.head(path, 1)
            except Exception:
                exists = False

            if not exists:
                print(f"[skip] {tname}: missing {path}")
                results.append({"table": tname, "status": "missing"})
                continue

            df = _stamp(_coerce_numeric(_read_csv(path)), cohort_id, run_id, fname)
            action = _merge(df, fqn, keys)
            rows = df.count()
            print(f"[{action}] {fqn} <- {fname}  ({rows:,} rows)")
            results.append({"table": tname, "status": action, "rows": rows})
        except Exception as e:
            print(f"[error] {tname}: {e}")
            results.append({"table": tname, "status": "error", "error": str(e)})
            raise

    import json
    return (json.dumps({
        "dataset_id": dataset_id,
        "run_id":     run_id,
        "results":    results,
    }))



try:
    _result = _run_body()
except Exception as _e:
    mark(run_id_root, cohort_id, stage, "FAILED", error=str(_e))
    raise
mark(run_id_root, cohort_id, stage, "SUCCEEDED")
mssparkutils.notebook.exit(_result)
